# 03 · Raw runs → aggregated results table

Rebuilds the aggregated table that every figure/table reads, from the per-run raw CSVs.

* **May 2026 campaign (default):** replays, step by step, the merge that produced
  `results_rigorous_20260523_115442.csv`, then checks the rebuild against the saved file.
  No mining is executed.
* **New campaign** (`FA_CAMPAIGN=<id>`): aggregates `results/runs/<id>/raw/*.csv` written by
  `02_run_grid.ipynb` with the same `baseline_runners.aggregate` and writes
  `results/runs/<id>/results_rigorous_<id>.csv`.

Both paths also write an **attempted-grid coverage table** (to `results/derived/`; saved evidence
under `results/campaign_2026-05/` is never rewritten). The aggregation drops a configuration
whose warm-up probe failed, so the coverage table is the only place timeouts/OOMs remain visible.

In [1]:
# ---- bundle configuration (same cell in every notebook) ----
import os, sys, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

BUNDLE = Path.cwd().resolve().parent          # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "baseline_runners.py").exists(), f"start Jupyter in <bundle>/notebooks (cwd={Path.cwd()})"
sys.path.insert(0, str(BUNDLE / "bench"))
DATA = Path(os.environ.get("FA_DATA_DIR", BUNDLE / "data"))
RESULTS = BUNDLE / "results"
DERIVED = RESULTS / "derived"                       # regenerated statistics, tables and figures

MAY = "campaign_2026-05"                           # the April/May 2026 GCE campaign
MAY_DIR = RESULTS / MAY
CAMPAIGN = os.environ.get("FA_CAMPAIGN", MAY) # set FA_CAMPAIGN=<id> to analyse a new run
CAMPAIGN_DIR = MAY_DIR if CAMPAIGN == MAY else RESULTS / "runs" / CAMPAIGN
AGG_CSV = (MAY_DIR / "results_rigorous_20260523_115442.csv" if CAMPAIGN == MAY
           else CAMPAIGN_DIR / f"results_rigorous_{CAMPAIGN}.csv")
LCM_CSV = MAY_DIR / "lcm_declat_results_20260523_055523.csv"
COST_CSV = (DERIVED / "cost_stats.csv") if (DERIVED / "cost_stats.csv").exists() else MAY_DIR / "cost_stats_20261022.csv"

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def rel(p):
    try:
        return Path(p).resolve().relative_to(BUNDLE).as_posix()
    except ValueError:
        return str(p)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print(f"campaign : {CAMPAIGN}\nagg csv  : {rel(AGG_CSV)}")

campaign : campaign_2026-05
agg csv  : results/campaign_2026-05/results_rigorous_20260523_115442.csv


In [2]:
import baseline_runners as br
RAW_DIR = CAMPAIGN_DIR / "raw"
for f in sorted(RAW_DIR.glob("*.csv")):
    print(f"{f.name:<52} {f.stat().st_size:>10,} B  sha256={sha256(f)[:16]}…")

results_rigorous_raw_20260420_171444.csv              2,104,434 B  sha256=0714c21ebffa3ab7…
results_rigorous_raw_20260423_090939.csv              7,222,069 B  sha256=c385690e4139c384…
results_rigorous_raw_20260425_014042.csv              1,106,595 B  sha256=e1508139abbb8d33…
results_rigorous_raw_20260428_205003.csv                966,245 B  sha256=b6f57f84b51d426f…
results_rigorous_raw_synth_20260425_172734.csv        5,392,170 B  sha256=55875e2330ff31e4…
results_rigorous_raw_synth_20260427_062019.csv        1,758,602 B  sha256=cad27b7d21941068…
results_rigorous_raw_synth_20260427_162201.csv        2,528,678 B  sha256=88ec21c6e085086a…
results_rigorous_raw_synth_20260429_030759.csv        1,118,711 B  sha256=df2a6f6c243840e4…


## 1 · Merge the raw runs
For the May 2026 campaign this is the merge order:

| step | file | role |
|---|---|---|
| v0 | `raw_20260423_090939` | real datasets, k=2…8 (v4 library: classic + baselines) |
| v0 | `raw_20260425_014042` | real datasets, k=8…9 |
| v0 | `raw_20260420_171444` | **only** its `efficient_apriori` rows |
| v0 | `raw_synth_20260425_172734`, `_20260427_162201`, `_20260427_062019` | synthetic k=2…6, 6…9, 6…7 |
| drop | — | every `fastapriori_fast` / `fastapriori_fast_1t` row of v0 |
| v5 | `raw_20260428_205003`, `raw_synth_20260429_030759` | fast (k=2…9) and fast_1t (k=2) reruns, v5 library |

In [3]:
if CAMPAIGN == MAY:
    read = lambda name: pd.read_csv(RAW_DIR / name)
    real, real9 = read("results_rigorous_raw_20260423_090939.csv"), read("results_rigorous_raw_20260425_014042.csv")
    ef = read("results_rigorous_raw_20260420_171444.csv")
    ef = ef[ef["method"] == "efficient_apriori"].copy()
    syn, syn2, syn3 = (read("results_rigorous_raw_synth_20260425_172734.csv"),
                       read("results_rigorous_raw_synth_20260427_162201.csv"),
                       read("results_rigorous_raw_synth_20260427_062019.csv"))
    v0 = pd.concat([real, real9, ef, syn, syn2, syn3])
    print("v0 rows:", len(v0))
    FAST = ["fastapriori_fast", "fastapriori_fast_1t"]
    v0_nofast = v0[~v0["method"].isin(FAST)].copy()
    print("v0 without fast rows:", len(v0_nofast))
    v5 = pd.concat([read("results_rigorous_raw_20260428_205003.csv"),
                    read("results_rigorous_raw_synth_20260429_030759.csv")])
    print("v5 fast reruns:", len(v5))
    raw = pd.concat([v0_nofast, v5]).reset_index(drop=True)
else:
    raw = pd.concat([pd.read_csv(f) for f in sorted(RAW_DIR.glob("*.csv"))], ignore_index=True)
print("merged raw rows:", len(raw))
raw.groupby(["method", "status"]).size().unstack(fill_value=0)

v0 rows: 19496
v0 without fast rows: 16824
v5 fast reruns: 2804
merged raw rows: 19628


status,ok,oom,timeout
method,,,
borgelt_apriori,2408,0,3
borgelt_eclat,2416,0,1
borgelt_fpgrowth,2416,0,1
efficient_apriori,156,0,1
fastapriori_classic,2172,12,14
fastapriori_fast,2424,20,0
fastapriori_fast_1t,360,0,0
pyfim_apriori,2404,2,2
pyfim_eclat,2404,4,0


### Check against the saved intermediate workbook (May 2026 campaign only)
The May 2026 analysis round-tripped the merged rows through `output_raw_fast_v5_29.4.xlsx`
and aggregated *that*. Rows are compared after sorting on the run key.

In [4]:
if CAMPAIGN == MAY:
    xl = pd.read_excel(MAY_DIR / "intermediate" / "output_raw_fast_v5_29.4.xlsx")
    key = ["dataset", "method", "k", "min_support", "run_id", "timestamp_utc"]
    a = raw.sort_values(key).reset_index(drop=True)
    b = xl.sort_values(key).reset_index(drop=True)
    assert a.shape == b.shape, (a.shape, b.shape)
    assert (a[key[:5]].astype(str).values == b[key[:5]].astype(str).values).all()
    num = [c for c in ["wall_s", "peak_rss_kb", "n_itemsets", "user_cpu_s"] if c in a]
    diff = {c: float(np.nanmax(np.abs(a[c].astype(float) - b[c].astype(float)))) for c in num}
    print("rows:", a.shape, "| max |csv - xlsx| per column:", diff)
    assert max(diff.values()) < 1e-9

rows: (19628, 56) | max |csv - xlsx| per column: {'wall_s': 5.684341886080802e-14, 'peak_rss_kb': 0.0, 'n_itemsets': 0.0, 'user_cpu_s': 0.0}


## 2 · Aggregate (`baseline_runners.aggregate`, unchanged)

In [5]:
agg = br.aggregate(raw, reference_method="fastapriori_fast")
print("aggregated configurations:", len(agg), "| columns:", agg.shape[1])
if CAMPAIGN == MAY:
    saved = pd.read_csv(AGG_CSV, low_memory=False)
    print("saved file sha256:", sha256(AGG_CSV))
    assert sha256(AGG_CSV) == "0f76d3232341a0b8d3670e752da58dc10efa9bbdd0c50f5972fdcf048720a03e"
    key = ["dataset", "method", "k", "min_support"]
    a = agg.sort_values(key).reset_index(drop=True)
    b = saved.sort_values(key).reset_index(drop=True)
    assert a.shape == b.shape and list(a.columns) == list(b.columns), (a.shape, b.shape)
    assert (a[key].astype(str).values == b[key].astype(str).values).all()
    worst = {}
    for c in a.columns:
        if pd.api.types.is_numeric_dtype(b[c]):
            x, y = a[c].astype(float).to_numpy(), b[c].astype(float).to_numpy()
            both_nan = np.isnan(x) & np.isnan(y)
            assert (np.isnan(x) == np.isnan(y)).all(), c
            rel_err = np.abs(x - y)[~both_nan] / np.maximum(1e-12, np.abs(y)[~both_nan])
            worst[c] = float(rel_err.max()) if rel_err.size else 0.0
        else:
            assert (a[c].astype(str).values == b[c].astype(str).values).all(), c
    top = sorted(worst.items(), key=lambda kv: -kv[1])[:5]
    print("largest relative differences (floating-point summation order):", top)
    assert top[0][1] < 1e-9
    print("REBUILD MATCHES the saved results_rigorous_20260523_115442.csv")
else:
    agg.to_csv(AGG_CSV, index=False)
    print("wrote", rel(AGG_CSV))

aggregated configurations: 4891 | columns: 232


saved file sha256: 0f76d3232341a0b8d3670e752da58dc10efa9bbdd0c50f5972fdcf048720a03e


largest relative differences (floating-point summation order): [('wall_s_iqr', 2.7483898543486138e-12), ('wall_s_range', 1.32870197750247e-12), ('peak_rss_kb_cv', 9.444921403024832e-13), ('wall_s_std', 9.141149230829825e-13), ('minor_page_faults_cv', 8.948207595422627e-13)]
REBUILD MATCHES the saved results_rigorous_20260523_115442.csv


## 3 · Attempted-grid coverage (failures included)
One row per attempted (dataset, method, k, support): the warm-up probe status, the number of
successful timed repetitions, and whether the configuration appears in the aggregated table.
For the May 2026 campaign this file is also the **grid used for a rerun** (`02_run_grid.ipynb`).

In [6]:
key = ["dataset", "method", "k", "min_support"]
probe = raw[raw["warmup"].astype(str).str.lower().isin(["true", "1"])]
timed = raw[~raw.index.isin(probe.index)]
cov = (probe.groupby(key)["status"].first().rename("probe_status").to_frame()
       .join(timed[timed.status == "ok"].groupby(key).size().rename("n_ok_timed"))
       .join(timed[timed.status != "ok"].groupby(key)["status"].first().rename("timed_failure")))
cov["n_ok_timed"] = cov["n_ok_timed"].fillna(0).astype(int)
in_agg = agg.set_index(key).index
cov["in_aggregate"] = cov.index.isin(in_agg)
cov = cov.reset_index()
DERIVED.mkdir(parents=True, exist_ok=True)
cov.to_csv(DERIVED / f"coverage_{CAMPAIGN}.csv", index=False, lineterminator="\n")
print("attempted configurations:", len(cov), "| in aggregate:", int(cov.in_aggregate.sum()))
print(cov.groupby(["method", "probe_status"]).size().unstack(fill_value=0))
cov[~cov.in_aggregate].groupby(["probe_status", "method"]).size()

attempted configurations: 4955 | in aggregate: 4891
probe_status          ok  oom  timeout
method                                
borgelt_apriori      602    0        3
borgelt_eclat        604    0        1
borgelt_fpgrowth     604    0        1
efficient_apriori     39    0        1
fastapriori_classic  543   12       14
fastapriori_fast     606   20        0
fastapriori_fast_1t   90    0        0
pyfim_apriori        601    2        2
pyfim_eclat          601    4        0
pyfim_fpgrowth       601    4        0


probe_status  method             
oom           fastapriori_classic    12
              fastapriori_fast       20
              pyfim_apriori           2
              pyfim_eclat             4
              pyfim_fpgrowth          4
timeout       borgelt_apriori         3
              borgelt_eclat           1
              borgelt_fpgrowth        1
              efficient_apriori       1
              fastapriori_classic    14
              pyfim_apriori           2
dtype: int64

In [7]:
if CAMPAIGN == MAY:
    grid = cov[key + ["probe_status"]].rename(columns={"probe_status": "may_probe_status"})
    grid.to_csv(DERIVED / "grid_2026-05.csv", index=False, lineterminator="\n")
    # the shipped (frozen) copies used by 02 and 07 must equal what the raw files give
    for name, frame in [("grid_2026-05.csv", grid), (f"coverage_{CAMPAIGN}.csv", cov)]:
        shipped = pd.read_csv(MAY_DIR / name)
        assert shipped.astype(str).equals(frame.reset_index(drop=True).astype(str)), name
    print("coverage and grid equal the shipped copies;", len(grid), "cells")
    print(grid.groupby("method").size())

coverage and grid equal the shipped copies; 4955 cells
method
borgelt_apriori        605
borgelt_eclat          605
borgelt_fpgrowth       605
efficient_apriori       40
fastapriori_classic    569
fastapriori_fast       626
fastapriori_fast_1t     90
pyfim_apriori          605
pyfim_eclat            605
pyfim_fpgrowth         605
dtype: int64
